In [1]:
# Import libraries
import pandas as pd
import gridstatus

In [2]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv(r'eia_authentication.env')

True

In [3]:
eia = gridstatus.EIA() # Read API key

In [4]:
# Pick fetch interval
start = "2023-12-31"
end = "2025-01-02"

In [5]:
# Load demand data for CAISO and ERCOT
demand = eia.get_dataset(
    "electricity/rto/region-data",
    start=start, end=end,
    facets={"respondent": ["CISO", "ERCO"]}
)

#print(demand.columns.tolist(), demand.head())

In [6]:
# Filter to relevant columns and separate by region
caiso_demand = (
    demand.loc[demand["Respondent"] == "CISO"]
    .set_index("Interval Start")["Load"]
    .sort_index()
    .copy()
)

ercot_demand = (
    demand.loc[demand["Respondent"] == "ERCO"]
    .set_index("Interval Start")["Load"]
    .sort_index()
    .copy()
)

In [7]:
# Load generation data for CAISO and ERCOT
gen = eia.get_dataset(
    "electricity/rto/fuel-type-data",
    start=start, end=end,
    facets={"respondent": ["CISO", "ERCO"]},
)

#print(gen.columns.tolist(), gen.head())

In [8]:
# Filter to relevant columns and separate by region
caiso_gen = (
    gen.loc[gen["Respondent"] == "CISO"]
    .set_index("Interval Start")[["Solar", "Wind"]]
    .sort_index()
    .copy()
)

ercot_gen = (
    gen.loc[gen["Respondent"] == "ERCO"]
    .set_index("Interval Start")[["Solar", "Wind"]]
    .sort_index()
    .copy()
)

In [9]:
# Assemble DataFrames for cleaning
full = pd.date_range(start, end, freq="h", tz="UTC", inclusive="left", name="Interval Start") # Set common index for data

caiso = pd.merge(caiso_demand, caiso_gen, left_index=True, right_index=True, how="outer").reindex(full)
ercot = pd.merge(ercot_demand, ercot_gen, left_index=True, right_index=True, how="outer").reindex(full)

In [10]:
# Save as CSVs
caiso.to_csv("data/caiso_raw.csv")
ercot.to_csv("data/ercot_raw.csv")

In [11]:
# Check for missing hours
def missing_hours(df, name, start=start, end=end):
    """Check for duplicate, missing, and NaN hours for an hourly UTC-indexed DataFrame or Series."""
    df = df.to_frame() if isinstance(df, pd.Series) else df
    full = pd.date_range(start, end, freq="h", tz="UTC", inclusive="left")

    dups = df.index[df.index.duplicated()]
    missing = full.difference(df.index)
    nan_rows = df.index[df.isna().any(axis=1)]
    gaps = missing.union(nan_rows).sort_values()

    print(f"---- {name}: {len(df)} rows, expected {len(full)} ----")
    print(f"duplicates: {len(dups)} | missing rows: {len(missing)} | NaN rows: {len(nan_rows)}")
    print("NaNs by column:", df.isna().sum()[lambda s: s > 0].to_dict())

    if len(gaps):
        # Consecutive missing hours share a run ID
        s = gaps.to_series()
        run_id = s.diff().dt.total_seconds().ne(3600).cumsum()
        runs = (s.groupby(run_id.values)
                .agg(start="first", end="last", hours="count"))
        print(runs.to_string(index=False))
    return gaps

# Check merged tables for gaps
for name, s in [("CAISO", caiso),
                ("ERCOT", ercot)]:
    missing_hours(s, name)
    print("")

---- CAISO: 8832 rows, expected 8832 ----
duplicates: 0 | missing rows: 0 | NaN rows: 24
NaNs by column: {'Solar': 24, 'Wind': 24}
                    start                       end  hours
2024-11-02 07:00:00+00:00 2024-11-03 06:00:00+00:00     24

---- ERCOT: 8832 rows, expected 8832 ----
duplicates: 0 | missing rows: 0 | NaN rows: 4
NaNs by column: {'Solar': 4}
                    start                       end  hours
2024-07-16 03:00:00+00:00 2024-07-16 03:00:00+00:00      1
2024-11-08 02:00:00+00:00 2024-11-08 02:00:00+00:00      1
2024-11-08 04:00:00+00:00 2024-11-08 05:00:00+00:00      2



In [12]:
# Pull CAISO generation data from gridstatus (as backup to fill in EIA's missing generation day)
caiso_gs = gridstatus.CAISO()

# Dates interpreted in Pacific time; doesn't include end time
load = caiso_gs.get_load(start="2024-11-01", end="2024-11-04")
fuel = caiso_gs.get_fuel_mix(start="2024-11-01", end="2024-11-04")
print(load.columns.tolist())
print(fuel.columns.tolist())

# Aggregate 5-minute data into hourly means, labeled by interval start in UTC (matching EIA data)
def to_hourly(df, cols):
    out = df.set_index("Interval Start")[cols]
    out.index = out.index.tz_convert("UTC")
    return out.resample("1h").mean()

caiso_fill = pd.concat(
    [to_hourly(load, ["Load"]), to_hourly(fuel, ["Solar", "Wind"])], axis=1
)

# Save to CSV
caiso_fill.to_csv("data/caiso_gridstatus_2024-11-01_03.csv")

100%|██████████| 3/3 [00:01<00:00,  3.00it/s]

['Time', 'Interval Start', 'Interval End', 'Load']
['Time', 'Interval Start', 'Interval End', 'Solar', 'Wind', 'Geothermal', 'Biomass', 'Biogas', 'Small Hydro', 'Coal', 'Nuclear', 'Natural Gas', 'Large Hydro', 'Batteries', 'Imports', 'Other']
